In [4]:
!pip install torch transformers pandas numpy matplotlib seaborn scikit-learn

In [ ]:

# BERT Customer Feedback Classifier

# 1. Import libraries

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from torch.utils.data import Dataset, DataLoader


# 2. Load dataset

data = pd.read_csv("IMDB_cleaned.csv")

print("Dataset loaded successfully!")
print(data.head())

print("\nDataset Shape:", data.shape)


# 3. Explore dataset

print("\nSentiment Distribution:")
print(data["sentiment"].value_counts())

print("\nMissing Values:")
print(data.isnull().sum())


# 4. Remove missing values and duplicate reviews

data = data.dropna()
data = data.drop_duplicates()

# Convert sentiment into numbers
data["sentiment"] = data["sentiment"].map({
    "negative": 0,
    "positive": 1
})

print("\nData preprocessing completed!")


# 5. Split dataset

X = data["review"]
y = data["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))


# 6. Load BERT tokenizer

tokenizer = BertTokenizer.from_pretrained(
    "bert-base-uncased"
)

print("\nBERT tokenizer loaded!")


# 7. Tokenize reviews

train_tokens = tokenizer(
    X_train.tolist(),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

test_tokens = tokenizer(
    X_test.tolist(),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print("\nTokenization completed!")


# 8. Create dataset class

class ReviewDataset(Dataset):

    def __init__(self, tokens, labels):
        self.tokens = tokens
        self.labels = torch.tensor(
            labels.values, dtype=torch.long
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):

        return {
            "input_ids": self.tokens["input_ids"][index],
            "attention_mask": self.tokens["attention_mask"][index],
            "labels": self.labels[index]
        }


# 9. Prepare DataLoader

train_dataset = ReviewDataset(
    train_tokens, y_train
)

test_dataset = ReviewDataset(
    test_tokens, y_test
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8
)

print("\nData preparation completed!")


# 10. Load pretrained BERT model

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

print("\nBERT model loaded!")
print("Using device:", device)


# 11. Set training parameters

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5
)

epochs = 2


# 12. Train the model

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for batch in train_loader:

        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        optimizer.zero_grad()

        output = model(**batch)

        loss = output.loss
        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    print(
        "Epoch:", epoch + 1,
        "Loss:", round(average_loss, 4)
    )

print("\nModel training completed!")


# 13. Evaluate the model

model.eval()

y_true = []
y_pred = []

with torch.no_grad():

    for batch in test_loader:

        labels = batch["labels"].numpy()

        inputs = {
            "input_ids": batch["input_ids"].to(device),
            "attention_mask": batch["attention_mask"].to(device)
        }

        output = model(**inputs)

        predictions = torch.argmax(
            output.logits, dim=1
        ).cpu().numpy()

        y_true.extend(labels)
        y_pred.extend(predictions)


# 14. Calculate evaluation metrics

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print("\nModel Evaluation Results")

print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1 Score:", round(f1, 4))


# 15. Classification report

print("\nClassification Report:")

print(
    classification_report(
        y_true,
        y_pred,
        target_names=["Negative", "Positive"]
    )
)


# 16. Confusion matrix

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("BERT Confusion Matrix")

plt.show()


# 17. Predict new customer feedback

def predict_feedback(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    model.eval()

    with torch.no_grad():

        output = model(**inputs)

    prediction = torch.argmax(
        output.logits, dim=1
    ).item()

    if prediction == 1:
        print("\nFeedback:", text)
        print("Prediction: Positive")
    else:
        print("\nFeedback:", text)
        print("Prediction: Negative")


# 18. Test new feedback

predict_feedback(
    "The service was excellent and I am very happy."
)

predict_feedback(
    "The product quality is very poor and disappointing."
)

predict_feedback(
    "I really enjoyed this experience."
)

predict_feedback(
    "Very bad service. I am completely disappointed."
)


# 19. Save trained model

model.save_pretrained("saved_bert_model")

tokenizer.save_pretrained("saved_bert_model")

print("\nModel saved successfully!")


# 20. Final result

print("\n===================================")
print("BERT CUSTOMER FEEDBACK CLASSIFIER")
print("===================================")

print("Dataset: IMDb 50K Reviews")
print("Model: BERT Base Uncased")
print("Training: Completed")
print("Evaluation: Completed")
print("Prediction: Completed")
print("Model Saving: Successful")

print("===================================")


Dataset loaded successfully!
                                              review sentiment  \
0  $25,000 Pyramid Clues: Deep Blue Sea. Tremors....  negative   
1  0.5/10. This movie has absolutely nothing good...  negative   
2  0*'s Christian Slater, Tara Reid, Stephen Dorf...  negative   
3  102 Dalmatians (2000, Dir. Kevin Lima) <br /><...  negative   
4  102 DALMATIANS [Walt Disney]: I wasn't a fan o...  negative   

                                      cleaned_review  sentiment_label  
0  pyramid clue deep blue sea tremor slither eigh...                0  
1  movie absolutely nothing good acting among wor...                0  
2  christian slater tara reid stephen dorff frank...                0  
3  dalmatian dir kevin lima believed cured cruell...                0  
4  dalmatian walt disney wasnt fan previous insta...                0  

Dataset Shape: (49396, 4)

Sentiment Distribution:
sentiment
negative    24698
positive    24698
Name: count, dtype: int64

Missing Values:
r

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



BERT model loaded!
Using device: cpu
